In [1]:
%pip install -q mediapipe

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install mediapipe opencv-python

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   - -------------------------------------- 1.8/40.2 MB 7.2 MB/s eta 0:00:06
   ----- ---------------------------------- 5.8/40.2 MB 12.8 MB/s eta 0:00:03
   -------- ------------------------------- 8.1/40.2 MB 12.4 MB/s eta 0:00:03
   ------------ --------------------------- 12.3/40.2 MB 14.2 MB/s eta 0:00:02
   ---------------- ----------------------- 16.5/40.2 MB 15.1 MB/s eta 0:00:02
   ------------------- -------------------- 19.7/40.2 MB 15.1 MB/s eta 0:00:02
   ---------------------- ----------------- 22.8/40.2 MB 15.8 MB/s eta 0:00:02
   -------------------------- ------------- 27.0/40.2 MB 15.7 MB/s eta 0:00:01
   ------------------------------- -------- 31.2/40.2 MB 16.1 MB/s eta 0:00:01
   ----------------------------------- ---- 35.4/40.2 MB 16.5 MB/s eta 0:00:01
   -------------------------------------- - 38.5/40.2 MB 16.2 MB/s eta 0:0

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\sh210\\AppData\\Local\\anaconda3\\envs\\ml\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# مسار الموديل
MODEL_PATH = "hand_landmarker.task"

# إعدادات الموديل
BaseOptions = python.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2
)

# فتح الكاميرا
cap = cv2.VideoCapture(0)

with HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # OpenCV → RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        # التوقيت مطلوب في وضع الفيديو
        timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))

        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        # رسم النقاط
        if result.hand_landmarks:
            for hand_landmarks in result.hand_landmarks:
                for lm in hand_landmarks:
                    h, w, _ = frame.shape
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

        cv2.imshow("Hand Landmarker", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 